# Randy Hield
# 6434137

# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)
Students work through this unified exercise in the VRB or on paper, structured across the six developmental steps of Lecture 1:

1. **E1 · Opening Judgment (Data vs. Competence):**
   - If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5.
     

2. **E2 · Changed Case Prediction (Physical Bounds):**
   - A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.
     

3. **E3 · The Commitment Decision (Warrant Gate):**
   - The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.
     

4. **E4 · The Five Questions of Understanding (T1–T5):**
   - Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.
     

5. **E5 · System Decomposition (Mapping to Architecture):**
   - List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.
     

6. **E6 · Transfer & Exit Ticket (Justified Adaptation):**
   - Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.

### Answers – Part 1

**E1 · Opening Judgment**

**No.** 100 successful videos show the robot succeeds under the conditions shown. They say nothing about untested conditions (another mug, a fuller cup, Eva moving), and failures may simply not have been filmed. 
**Confidence: 4/5**, 

**E2 · Changed Case Prediction**

**Prediction:** in the turn, sideways acceleration may make the coffee splash against the outer rim. The half-full mug had enough free space above the liquid to absorb the wave; a nearly full mug doesn't, so coffee spills, possibly hot near Eva.
**Untested assumption:** that fill level doesn't matter, which means that a speed and turn safe for a half-full mug are safe for any fill.

**E3 · Commitment Decision**

**Ask.** Observing can confirm that a white mug exists, but not that Eva accepts it. That is a grounding question only she can answer. Proceeding would fail the grounding check, and coffee poured into the wrong mug can't be undone.
**Evidence:** Eva's explicit confirmation that the white mug is acceptable.

**E4 · Five Questions**

- **Explain:** The robot paused once mid-route. Was that because of an obstacle, or because it predicted too much slosh?
- **Predict:** *If Eva is standing at her walking frame instead of sitting, can she still reach the usual handover height?* (changed condition)
- **Design:** Would placing the mug on her side table instead of handing it over reduce spill and collision risk?
- **Bound:** For which floors, fill levels and coffee temperatures does the carry stay within the spill limit?
- **Check:** Does the record show Eva accepted the white mug *before* the robot poured?

**E5 · System Decomposition**

| Module | Integration diagram box |
|---|---|
| Interpreting | interpret & ground |
| Perceiving | perceive (reading → belief) |
| Predicting | know & predict (digital twin) |
| Proposing | propose (System 1 · Learning) |
| Warranting | decide: warrant gate (System 2 · Warrant) |
| Acting | act & regulate (move, watch, repair) |
| Recording | record & justify |

All modules form the Manipulation Executive, which acts on the Body–World System. Flow: perceive → interpret & ground → propose → predict → warrant gate → act & regulate → record & justify. A failed check routes back (e.g. ask Eva) instead of to the body.

**E6 · Transfer & Exit Ticket**

**Revised action:** decelerate gently and stop with the mug level, re-observe Eva, choose a new handover point within her reach that doesn't block her walking frame, then approach slowly.
**Failure mechanisms:** a sudden stop sloshes hot coffee; collision if she steps into the path; handover out of reach, so she leans or stumbles.
**Evidence before committing:** Eva has been stationary for ≥ 2 s, the new handover pose is reachable for her with ≥ 0.5 m path clearance, and the twin predicts slosh below the mug's limit. (Thresholds are assumptions.)

## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."

---

In [ ]:
#!/usr/bin/env python3
"""Embodied Cognitive Architecture (completed for Tutorial 1, Part 2).
Every change to the original script is marked with a [Task x.y] comment.
"""

from dataclasses import dataclass, field
from typing import Dict, Any, Tuple, Optional, List
from collections import Counter
import math
import random
import time

@dataclass(frozen=True)
class GoalSpecification:
    """Formalized, concrete goal specification derived via semantic interpretation."""
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float

@dataclass
class ParameterPacket:
    """System 1 (The Proposer): Flexible sampler proposing execution parameters."""
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    source_generator: str = "stochastic_sampler"


class Proposer:
    """[Task 1.2] System 1 (propose): Gaussian sampler of execution parameters.

    Each call draws a NEW candidate around a 'habitual' value, so a rejected proposal
    can be followed by a different one. Means were derived from the twin's own formulas:
      - force rises sharply above ~0.33 m/s (bottoming out, Task 2.1) -> speed mean 0.28 m/s
      - wave_mm ~ 42 * tilt must stay below 1.5 mm               -> tilt mean 0.025 rad/s
    Samples are clipped to physically meaningful ranges.
    """
    def __init__(self, v_mean: float = 0.28, v_std: float = 0.03,
                 tilt_mean: float = 0.025, tilt_std: float = 0.005,
                 seed: Optional[int] = None):
        self.v_mean, self.v_std = v_mean, v_std
        self.tilt_mean, self.tilt_std = tilt_mean, tilt_std
        self.rng = random.Random(seed)

    def propose(self, goal: GoalSpecification) -> ParameterPacket:
        v = self.rng.gauss(self.v_mean, self.v_std)
        tilt = self.rng.gauss(self.tilt_mean, self.tilt_std)
        v = min(max(v, 0.05), goal.max_nav_velocity_mps)     # never exceed the goal's limit
        tilt = min(max(tilt, 0.005), 0.2)
        return ParameterPacket(grasp_pose=(0.45, 0.12, 0.30),
                               max_velocity_mps=round(v, 3),
                               tilt_rate_radps=round(tilt, 4),
                               source_generator="gaussian_sampler")


@dataclass
class Observation:
    """Raw, noisy sensor measurements at time t (distinct from internal belief)."""
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str

@dataclass
class BeliefState:
    """The robot's current internal belief state (updated via perception)."""
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True
    # Real-time environmental parameters maintained internally
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"  # 'low', 'medium', 'high'
    preferred_cup: str = "blue_ceramic_mug"
    cup_accepted_by_eva: bool = True      # grounding: has Eva accepted this cup? (lecture slide 23)
    battery_level_pct: float = 80.0       # [Task 4.2] current state of charge
    battery_capacity_wh: float = 200.0    # [Task 4.1] total battery capacity (assumed)
    route_length_m: float = 12.0          # [Task 4.1] kitchen -> Eva -> dock (assumed)


class InterpretationSystem:
    """Uses a semantic knowledge base to map vague instructions and context
    into precise physical goal parameters (GoalSpecification).
    """
    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        print(f"\n[INTERPRETATION] Analyzing instruction: '{raw_utterance}' in context of Eva (Position: {belief.eva_position}, Room Busyness: {belief.room_busyness})")

        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(cup_type, {"max_force": 5.0, "max_wave": 2.0})

        busyness_factor = self.kb["room_busyness_limits"].get(belief.room_busyness, 0.5)
        max_nav_vel = 1.2 * busyness_factor

        is_urgent = "schnell" in raw_utterance.lower() or "fast" in raw_utterance.lower()
        time_frame = 30.0 if is_urgent else 60.0
        target_volume = 15.0 if not is_urgent else 12.0

        goal = GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=target_volume,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=max_nav_vel,
            time_frame_seconds=time_frame
        )

        print(f" -> Derived Goal: Object={goal.target_object}, MaxForce={goal.max_allowable_force_n}N, MaxNavVel={goal.max_nav_velocity_mps}m/s, TimeFrame={goal.time_frame_seconds}s")
        return goal


class SensingSystem:
    """Simulates real-time noisy hardware sensors emitting raw observations."""
    def __init__(self):
        pass

    def sense(self, true_ambient_temp: float, true_friction: float, eva_pos: str, busyness: str) -> Observation:
        noisy_temp = true_ambient_temp + random.uniform(-0.5, 0.5)
        noisy_friction = true_friction + random.uniform(-0.05, 0.05)

        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(noisy_temp, 2),
            measured_friction=round(noisy_friction, 2),
            eva_position=eva_pos,
            room_busyness=busyness
        )


class PerceptionSystem:
    """Estimator module that updates the internal BeliefState using raw Observations."""
    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness


class SemanticDigitalTwin:
    """System 2 (Semantic Digital Twin): Imperfect models with sim-to-real epistemic gaps."""
    # [Task 2.1] dual-stage material compliance
    CONTACT_TIME_S = 0.06        # ASSUMPTION: time the contact compresses before the controller stops
    BOTTOM_OUT_DEPTH_M = 0.02    # beyond this depth the pouch presses on the rigid internal fixture
    STIFFENING_FACTOR = 1.8
    # [Task 3.1] viscosity-temperature coupling
    REF_TEMP_C = 20.0
    LINEAR_DROP_PER_C = 0.01     # 1 % drop per degC above 20 degC
    ARRHENIUS_B_K = 859.0        # chosen so the Arrhenius slope is also ~1 %/degC near 20 degC
    # [Task 4.1] energy model (all values are illustrative assumptions)
    P_BASE_W = 120.0             # computers, sensors, standby
    P_DRIVE_NOMINAL_W = 60.0     # base motors at 0.5 m/s, scales with v^2
    P_ARM_W = 50.0               # arm during grasp and handover
    MANIPULATION_TIME_S = 60.0

    def __init__(self, belief_state: BeliefState, epistemic_noise: bool = True,
                 viscosity_model: str = "linear"):
        self.belief = belief_state
        self.epistemic_noise = epistemic_noise     # False -> deterministic predictions for tests
        self.viscosity_model = viscosity_model     # "linear" or "arrhenius"

    # ---- [Task 2.1] ----
    def contact_depth(self, velocity_mps: float) -> float:
        """Penetration depth = approach velocity x compression time (assumption)."""
        return velocity_mps * self.CONTACT_TIME_S

    def stiffness(self, depth_m: float) -> float:
        """Stage 1: soft pouch. Stage 2 (depth > 2 cm): jumps by x1.8 at bottom-out and
        keeps growing exponentially with every extra cm of compression."""
        k0 = 12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)
        if depth_m <= self.BOTTOM_OUT_DEPTH_M:
            return k0
        excess = (depth_m - self.BOTTOM_OUT_DEPTH_M) / self.BOTTOM_OUT_DEPTH_M
        return k0 * self.STIFFENING_FACTOR ** (1.0 + excess)

    # ---- [Task 3.1] ----
    def effective_viscosity(self, temp_c: float) -> float:
        mu_ref = self.belief.fluid_viscosity_pas
        if self.viscosity_model == "arrhenius":
            t_k, t_ref_k = temp_c + 273.15, self.REF_TEMP_C + 273.15
            return mu_ref * math.exp(self.ARRHENIUS_B_K * (1.0 / t_k - 1.0 / t_ref_k))
        drop = max(0.0, temp_c - self.REF_TEMP_C) * self.LINEAR_DROP_PER_C
        return mu_ref * max(0.05, 1.0 - drop)          # never let viscosity reach zero

    # ---- [Task 4.1] ----
    def predict_battery_drain_pct(self, params: ParameterPacket) -> float:
        v = max(params.max_velocity_mps, 0.05)
        t_drive = self.belief.route_length_m / v
        p_drive = self.P_DRIVE_NOMINAL_W * (v / 0.5) ** 2
        energy_j = (self.P_BASE_W * (t_drive + self.MANIPULATION_TIME_S)
                    + p_drive * t_drive + self.P_ARM_W * self.MANIPULATION_TIME_S)
        energy_wh = energy_j / 3600.0
        return 100.0 * energy_wh / self.belief.battery_capacity_wh

    def predict_properties(self, params: ParameterPacket, goal: GoalSpecification) -> Dict[str, float]:
        depth = self.contact_depth(params.max_velocity_mps)                     # [Task 2.1]
        stiffness_coefficient = self.stiffness(depth)                           # [Task 2.1]
        predicted_force = stiffness_coefficient * (params.max_velocity_mps ** 1.5)

        effective_viscosity = self.effective_viscosity(self.belief.ambient_temperature_c)  # [Task 3.1]
        radius = 0.05
        wave_height_m = (params.tilt_rate_radps * radius) / effective_viscosity
        predicted_wave = wave_height_m * 1000.0

        # Injecting minor epistemic simulation error (Sim-to-Real gap)
        epistemic_force_error = random.uniform(-0.2, 0.3) if self.epistemic_noise else 0.0
        epistemic_wave_error = random.uniform(-0.1, 0.2) if self.epistemic_noise else 0.0

        return {
            "predicted_force_n": round(max(0.0, predicted_force + epistemic_force_error), 2),
            "predicted_wave_mm": round(max(0.0, predicted_wave + epistemic_wave_error), 2),
            "predicted_nav_velocity": min(params.max_velocity_mps, goal.max_nav_velocity_mps),
            "contact_depth_m": round(depth, 4),                                  # [Task 2.1]
            "stiffness_coefficient": round(stiffness_coefficient, 2),            # [Task 2.1]
            "effective_viscosity_pas": round(effective_viscosity, 4),            # [Task 3.1]
            "predicted_battery_drain_pct": round(self.predict_battery_drain_pct(params), 2),  # [Task 4.1]
        }


class WarrantGate:
    """Validates predicted properties against the interpreted GoalSpecification."""
    MAX_DRAIN_PCT = 15.0     # [Task 4.1] a single task may use at most 15 % of total capacity
    RESERVE_PCT = 10.0       # [Task 4.2] charge that must remain afterwards to reach the dock

    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt

    def validate(self, params: ParameterPacket, goal: GoalSpecification, belief: BeliefState) -> Dict[str, Any]:
        predictions = self.sdt.predict_properties(params, goal)
        drain = predictions["predicted_battery_drain_pct"]

        checks = {
            "grounded": belief.vial_ready and belief.pouch_present and belief.cup_accepted_by_eva,
            "feasible": params.max_velocity_mps > 0.0 and params.tilt_rate_radps > 0.0,
            "safe": (predictions["predicted_force_n"] <= goal.max_allowable_force_n) and
                    (predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm) and
                    (predictions["predicted_nav_velocity"] <= goal.max_nav_velocity_mps),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled,
            # [Task 4.1] 7th criterion
            "energetically_viable": (drain < self.MAX_DRAIN_PCT) and
                                    (belief.battery_level_pct - drain >= self.RESERVE_PCT),
        }

        all_passed = all(checks.values())
        failed_criterion = next((name for name, passed in checks.items() if not passed), None)

        return {
            "passed": all_passed,
            "failed_criterion": failed_criterion,
            "predictions": predictions,
            "warrant_checks": checks
        }


class ExecutionEngine:
    """Manages physical execution and constructs the epistemic audit trail."""
    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params: ParameterPacket, goal: GoalSpecification, warrant_result: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n[ACTUATOR] Executing motion in physical world: v_max={params.max_velocity_mps} m/s")
        time.sleep(0.1)

        # Real-world divergence check
        predictions = warrant_result["predictions"]
        actual_force = predictions["predicted_force_n"] + random.uniform(-0.3, 0.6)
        actual_wave = predictions["predicted_wave_mm"] + random.uniform(-0.2, 0.5)

        spill_occurred = (actual_force > goal.max_allowable_force_n) or (actual_wave > goal.max_slosh_wave_mm)
        status = "FAILURE" if spill_occurred else "SUCCESS"

        execution_outcome = {
            "actual_force_n": round(actual_force, 2),
            "actual_wave_mm": round(actual_wave, 2),
            "actual_volume_transferred_ml": goal.target_volume_ml if not spill_occurred else goal.target_volume_ml - 3.0,
            "spill_occurred": spill_occurred,
            "status": status
        }
        return execution_outcome

    def record_trace(self, asked_utterance: str, goal: GoalSpecification, belief: BeliefState,
                     proposal: ParameterPacket, warrant_result: Dict[str, Any],
                     observation: Observation, committed: bool, execution_outcome: Optional[Dict[str, Any]] = None,
                     decision: str = "") -> None:
        record = {
            "timestamp": time.time(),
            "asked": {
                "raw_utterance": asked_utterance,
                "interpreter_output": {
                    "target": goal.target_object,
                    "max_force_limit_n": goal.max_allowable_force_n,
                    "max_nav_velocity": goal.max_nav_velocity_mps
                }
            },
            "belief": {
                "eva_position": belief.eva_position,
                "room_busyness": belief.room_busyness,
                "ambient_temp_c": belief.ambient_temperature_c,
                "surface_friction": belief.surface_friction_coefficient,
                "battery_level_pct": belief.battery_level_pct,          # [Task 4.2]
            },
            "observed": {
                "timestamp": observation.timestamp,
                "measured_temp_c": observation.measured_temperature_c,
                "measured_friction": observation.measured_friction,
                "eva_position": observation.eva_position,
                "room_busyness": observation.room_busyness
            },
            "committed": {
                "action_approved": committed,
                "decision": decision,                                       # [Task 4.2] route taken
                "proposed_parameters": {
                    "max_velocity_mps": proposal.max_velocity_mps,
                    "tilt_rate_radps": proposal.tilt_rate_radps
                },
                "warrant_verdict": warrant_result
            },
            "execution_outcome": execution_outcome
        }
        self.audit_trail.append(record)


# [Task 1.2 / 4.2] The first failing check decides what happens next (lecture slide 23).
ROUTES = {
    "grounded": "ask_eva",                  # e.g. white mug not yet accepted
    "feasible": "resample",
    "safe": "resample",                     # a gentler proposal may pass
    "monitorable": "abort",
    "repairable": "abort",
    "recordable": "abort",
    "energetically_viable": "recharge",     # slower/faster motion will not fix a low battery
}


def run_episode(command: str, belief: BeliefState, world: Dict[str, Any], kb: Dict[str, Any],
                proposer: Proposer, sdt: SemanticDigitalTwin, executive: ExecutionEngine,
                max_attempts: int = 10, verbose: bool = True) -> Dict[str, Any]:
    """One cognitive decision cycle: sense -> perceive -> interpret -> (propose -> warrant)* -> act -> record."""
    log = print if verbose else (lambda *a, **k: None)
    observation = SensingSystem().sense(world["temp_c"], world["friction"], world["eva_pos"], world["busyness"])
    PerceptionSystem(belief).update(observation)
    if verbose:
        goal = InterpretationSystem(kb).interpret(command, belief)
    else:
        import contextlib, io
        with contextlib.redirect_stdout(io.StringIO()):
            goal = InterpretationSystem(kb).interpret(command, belief)
    gate = WarrantGate(sdt)

    for attempt in range(1, max_attempts + 1):
        proposal = proposer.propose(goal)
        warrant = gate.validate(proposal, goal, belief)
        if warrant["passed"]:
            log(f"Attempt {attempt}: v={proposal.max_velocity_mps} m/s, tilt={proposal.tilt_rate_radps} rad/s -> APPROVED")
            if verbose:
                outcome = executive.execute_action(proposal, goal, warrant)
            else:
                import contextlib, io
                with contextlib.redirect_stdout(io.StringIO()):
                    outcome = executive.execute_action(proposal, goal, warrant)
            executive.record_trace(command, goal, belief, proposal, warrant, observation,
                                   committed=True, execution_outcome=outcome, decision="execute")
            log(f"Execution outcome: {outcome}")
            return {"decision": "execute", "outcome": outcome, "attempts": attempt}

        failed = warrant["failed_criterion"]
        route = ROUTES[failed]
        executive.record_trace(command, goal, belief, proposal, warrant, observation,
                               committed=False, decision=route)
        log(f"Attempt {attempt}: v={proposal.max_velocity_mps} m/s, tilt={proposal.tilt_rate_radps} rad/s "
            f"-> REJECTED (failed: {failed}) -> route: {route}")
        if route == "ask_eva":
            log("[EXECUTIVE] Asking: 'Your blue mug is in the dishwasher. Is the white one all right?'")
            return {"decision": route, "outcome": None, "attempts": attempt}
        if route == "recharge":
            log(f"[EXECUTIVE] Battery {belief.battery_level_pct}% is too low for this task "
                f"(needs {warrant['predictions']['predicted_battery_drain_pct']}% + {WarrantGate.RESERVE_PCT}% reserve). "
                "Coffee delivery postponed -> navigating to charging dock. Telling Eva: 'I need to recharge first.'")
            return {"decision": route, "outcome": None, "attempts": attempt}
        if route == "abort":
            log("[EXECUTIVE] Cannot monitor/repair/record this action -> abort and inform Eva.")
            return {"decision": route, "outcome": None, "attempts": attempt}
        # route == "resample": loop and let System 1 propose again

    log("[EXECUTIVE] No warranted proposal found -> do not act; ask Eva / escalate.")
    return {"decision": "escalate", "outcome": None, "attempts": max_attempts}


KNOWLEDGE_BASE = {
    "cups": {
        "blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5},
        "plastic_cup": {"max_force": 7.0, "max_wave": 3.0}
    },
    "room_busyness_limits": {
        "low": 1.0,
        "medium": 0.6,
        "high": 0.3
    }
}

NORMAL_WORLD = {"temp_c": 21.5, "friction": 0.45, "eva_pos": "living_room", "busyness": "medium"}


# --- Execution Demonstration ---
if __name__ == "__main__":
    random.seed(42)
    belief_state = BeliefState(room_busyness="medium", preferred_cup="blue_ceramic_mug")
    sdt = SemanticDigitalTwin(belief_state)
    executive = ExecutionEngine()
    proposer = Proposer(seed=42)

    print("=== One full decision cycle: 'Bring me my coffee fast!' ===")
    result = run_episode("Bring me my coffee fast!", belief_state, NORMAL_WORLD, KNOWLEDGE_BASE,
                         proposer, sdt, executive)

    print(f"\n--- COMPLETE EPISODIC AUDIT TRAIL ({len(executive.audit_trail)} Entries) ---")
    for idx, trace in enumerate(executive.audit_trail, 1):
        status = trace['committed']['action_approved']
        outcome = trace.get('execution_outcome')
        outcome_str = f" | Outcome: {outcome['status']} (Spill: {outcome['spill_occurred']})" if outcome else ""
        print(f"Entry #{idx} | Approved: {status} | Decision: {trace['committed']['decision']}{outcome_str} | Command: '{trace['asked']['raw_utterance']}'")


## Questions (0.25 Pt x 4 + 0.5 Pt)

### Task 1: Architectural Mapping & Code Decomposition
* **Question 1.1:** Map the classes and methods implemented in your Python script directly onto the functional blocks of the lecture integration diagram. Explain how data flows between these components during a single cognitive decision cycle.
* **Question 1.2:** Complete the proposer class based on a probabilistic model of your choice (e.g., uniform, gaussian, mixture). Justify your choice.
  

### Task 2: Non-Linear Contact Mechanics and Material Compliance
In the base script, the contact force on the flexible pouch is modeled using a simplified power-law equation. 
* **Question 2.1:** Modify the `SemanticDigitalTwin.predict_properties` method to account for a dual-stage material compliance curve. If the contact depth exceeds $0.02 \, \text{m}$, the stiffness coefficient should increase exponentially by a factor of $1.8$ due to bottoming out against a rigid internal fixture.
* **Question 2.2:** Write a test case where the Proposer generates an aggressive approach velocity ($0.35 \, \text{m/s}$). Calculate the predicted force and verify that the Warrant Gate successfully intercepts and rejects the parameter packet before physical execution.

### Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling
Fluids in laboratory automation often exhibit temperature-dependent viscosity changes.
* **Question 3.1:** Enhance the fluid dynamics model so that ambient temperature fluctuations directly alter the effective fluid viscosity using an Arrhenius-type approximation or a linear thermal degradation coefficient ($1\% \, \text{drop per} \, ^\circ\text{C}$ above $20^\circ\text{C}$).
* **Question 3.2:** Simulate an environment where the ambient temperature rises to $28^\circ\text{C}$. How does this change the predicted wave height for a tilt rate of $0.15 \, \text{rad/s}$? Does the Warrant Gate trigger a safety violation?

### Task 4: Expanding the Warrant Gate Checklist
The current Warrant Gate evaluates six core criteria (*grounded, feasible, safe, monitorable, repairable, recordable*). 
* **Question 4.1:** Add a seventh criterion called **`energetically_viable`** to the Warrant Gate dictionary checks. It must evaluate whether the estimated battery drain for the requested trajectory stays below $15\%$ of total capacity.
* **Question 4.2:** Introduce a low-battery state into the `BeliefState` and demonstrate how a failed `energetically_viable` check redirects the system workflow to request a recharge action rather than proceeding with the coffee delivery.

### Task 5: Post-Hoc Epistemic Audit Analysis
A core pillar of trustworthy robotics is the ability to audit *why* a decision was made or rejected.
* **Question 5.1:** Write a post-processing function (`audit_analyzer(audit_trail)`) that parses the recorded episodic log. It should compute and print:
  1. The total number of rejected proposals versus approved executions.
  2. The most frequent warrant criterion responsible for rejection.
  3. The average variance between predicted forces and observed physical outcomes.
* **Question 5.2:** Discuss in your lab report: *Why does maintaining separate records for `asked`, `belief`, `committed`, and `observed` solve the epistemological gap that pure data-driven imitation learning cannot address?*

## Answers – Part 2

All changes to the architecture code above are marked with `[Task x.y]` comments. The cells below use those classes.

### Task 1.1 · Architectural mapping

| Integration diagram box | Code | Input → output |
|---|---|---|
| perceive (reading → belief) | `SensingSystem.sense()` → `PerceptionSystem.update()` | world → noisy `Observation` → updated `BeliefState` |
| interpret & ground | `InterpretationSystem.interpret()` | utterance + belief + knowledge base → `GoalSpecification` (numeric limits) |
| propose (System 1) | `Proposer.propose()` | goal → `ParameterPacket` (candidate speed / tilt) |
| know & predict (digital twin) | `SemanticDigitalTwin.predict_properties()` | packet + belief → predicted force, wave, battery drain |
| decide: warrant gate (System 2) | `WarrantGate.validate()` | predictions vs. goal → pass/fail + first failed criterion |
| act & regulate | `ExecutionEngine.execute_action()` | approved packet → observed outcome |
| record & justify | `ExecutionEngine.record_trace()` → `audit_trail` | asked / belief / committed / observed |
| Manipulation Executive | `run_episode()` | runs the cycle and routes on the failed check (`ROUTES`) |

**Data flow in one cycle:** world → observation → belief → goal → proposal → prediction → warrant. If the gate passes, the action is executed and recorded. If it fails, the first failed criterion picks the route: resample (`safe`), ask Eva (`grounded`), recharge (`energetically_viable`) or abort. Every rejection is recorded too.

**Gaps noticed:** (1) the original `grounded` check tested `vial_ready`/`pouch_present`, which say nothing about Eva's request; I added `cup_accepted_by_eva` (slide 23). (2) `execute_action` only acts. It does not *watch or repair* during the carry as the act & regulate box requires.

### Task 1.2 · Proposer

**Choice: Gaussian.** A Proposer must give a *new* candidate each time (a fixed one would be rejected forever). A uniform sampler wastes most tries on extreme values: only tilts below ≈ 0.033 rad/s keep the wave under 1.5 mm, which is < 20 % of the range. A Gaussian aims at a sensible "habitual" value and explores around it..

The comparison below backs this up: same warrant gate, 1000 proposals each.

In [ ]:
random.seed(1)
belief = BeliefState()
goal = GoalSpecification("blue_ceramic_mug", "living_room", 12.0, 4.5, 1.5, 0.72, 30.0)
gate = WarrantGate(SemanticDigitalTwin(belief))

gaussian = Proposer(seed=3)
uniform_rng = random.Random(5)
def uniform_propose(goal):
    return ParameterPacket((0.45, 0.12, 0.30),
                           round(uniform_rng.uniform(0.05, goal.max_nav_velocity_mps), 3),
                           round(uniform_rng.uniform(0.005, 0.2), 4), "uniform_sampler")

N = 1000
g_pass = sum(gate.validate(gaussian.propose(goal), goal, belief)["passed"] for _ in range(N))
u_pass = sum(gate.validate(uniform_propose(goal), goal, belief)["passed"] for _ in range(N))
print(f"Gaussian proposer: {g_pass/N:.0%} of proposals pass the warrant gate")
print(f"Uniform proposer : {u_pass/N:.0%} of proposals pass the warrant gate")

### Task 2.1 · Dual-stage compliance

The script has no contact depth, so I defined one (**assumption**): `depth = v × 0.06 s`, the compression before the controller stops.

- depth ≤ 0.02 m → soft pouch: `k = k0 = 12.5·(1 + 0.1·μ)`
- depth > 0.02 m → bottoming out on the rigid fixture: `k = k0 · 1.8^(1 + (depth − 0.02)/0.02)`. This is a ×1.8 jump at 2 cm that keeps growing exponentially with extra compression.

So bottoming out starts at v = 0.02 / 0.06 ≈ 0.33 m/s (see `contact_depth()` and `stiffness()` in the twin).

In [ ]:
belief = BeliefState()
twin = SemanticDigitalTwin(belief, epistemic_noise=False)   # deterministic for the table
goal = GoalSpecification("blue_ceramic_mug", "living_room", 12.0, 4.5, 1.5, 0.72, 30.0)

print(f"{'v [m/s]':>8} {'depth [m]':>10} {'k':>7} {'force [N]':>10}  stage")
for v in [0.20, 0.25, 0.30, 0.33, 0.35, 0.40]:
    p = twin.predict_properties(ParameterPacket((0.45, 0.12, 0.30), v, 0.025), goal)
    stage = "bottomed out" if p["contact_depth_m"] > twin.BOTTOM_OUT_DEPTH_M else "soft"
    print(f"{v:>8} {p['contact_depth_m']:>10} {p['stiffness_coefficient']:>7} {p['predicted_force_n']:>10}  {stage}")

### Task 2.2 · Test: aggressive approach at 0.35 m/s

**Calculation:** k0 = 12.5 × (1 + 0.45 × 0.1) = 13.06 · depth = 0.35 × 0.06 = 0.021 m > 0.02 → k = 13.06 × 1.8^1.05 = 24.21 → **F = 24.21 × 0.35^1.5 = 5.01 N > 4.5 N → reject.**
With the old single-stage model, F = 13.06 × 0.35^1.5 = 2.70 N, which would have **passed**. Only the new model catches it.

In [ ]:
def test_aggressive_velocity_is_rejected():
    random.seed(7)
    belief = BeliefState()
    goal = GoalSpecification("blue_ceramic_mug", "living_room", 12.0, 4.5, 1.5, 0.72, 30.0)
    aggressive = Proposer(v_mean=0.35, v_std=0.0, tilt_mean=0.025, tilt_std=0.0)  # always proposes 0.35 m/s
    packet = aggressive.propose(goal)
    assert packet.max_velocity_mps == 0.35

    # deterministic prediction
    exact = SemanticDigitalTwin(belief, epistemic_noise=False).predict_properties(packet, goal)
    print(f"Predicted force at 0.35 m/s: {exact['predicted_force_n']} N (limit {goal.max_allowable_force_n} N)")
    assert exact["predicted_force_n"] > goal.max_allowable_force_n

    # with sim-to-real noise: rejected in every one of 200 trials
    gate = WarrantGate(SemanticDigitalTwin(belief))
    for _ in range(200):
        w = gate.validate(packet, goal, belief)
        assert not w["passed"] and w["failed_criterion"] == "safe"

    # full pipeline: nothing is ever executed
    executive = ExecutionEngine()
    result = run_episode("Bring me my coffee fast!", belief, NORMAL_WORLD, KNOWLEDGE_BASE,
                         aggressive, SemanticDigitalTwin(belief), executive, max_attempts=3)
    assert result["decision"] == "escalate"
    assert all(not t["committed"]["action_approved"] for t in executive.audit_trail)
    assert all(t["execution_outcome"] is None for t in executive.audit_trail)
    print("\nTEST PASSED: 0.35 m/s is intercepted by the warrant gate before execution.")

test_aggressive_velocity_is_rejected()

## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.

---

# Submission guidelines

- Deadline: 27.10.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**. 